# LeetCode 898 — Bitwise ORs of Subarrays bằng Python

Link đề: https://leetcode.com/problems/bitwise-ors-of-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: OR chỉ **bật thêm bit**, nên các giá trị OR của những đoạn con **kết thúc tại cùng một vị trí** chỉ có tối đa `~31` giá trị khác nhau (chúng tạo thành dãy tăng dần theo bao hàm bit). Vì thế ta lăn một tập nhỏ qua từng vị trí. Oracle: **vét cạn mọi đoạn con** `[i, j]` tính OR bằng `functools.reduce`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số giá trị **khác nhau** của OR trên **mọi đoạn con liên tiếp** (không rỗng) của `arr`.
2. Nắm công cụ Python: toán tử `|`, `set`, `functools.reduce`, `bit_length`, duyệt bit `(x >> b) & 1`.
3. Đi từ **vét cạn mọi đoạn con (`O(n²)`)** → **tập OR của các đoạn kết thúc tại `i` (lăn tập)** → **danh sách tăng dần khử trùng liền kề** → **"vị trí gần nhất có bit `b`"**.
4. Hiểu mấu chốt: với điểm cuối `i` cố định, khi điểm đầu lùi dần OR **không giảm** (chỉ bật thêm bit) nên có tối đa `31` giá trị khác nhau (mỗi lần đổi giá trị phải bật ít nhất một bit mới, mà số bit `≤ 30`).
5. Tránh lỗi kinh điển: dùng `+`/`&` thay vì `|`, đếm số đoạn con thay vì số giá trị **khác nhau**, quên thêm chính `arr[i]`, khởi tạo tập đáp án chứa `0` khi mảng không có đoạn nào OR bằng `0`, và nhầm giữa tập "kết thúc tại `i`" với tập toàn cục.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Trả về **số lượng giá trị khác nhau** của phép OR theo bit trên **mọi đoạn con khác rỗng** của `arr`.

OR của một đoạn con là OR của tất cả phần tử trong nó; đoạn một phần tử có OR bằng chính phần tử đó.

**Ví dụ 1:** `arr = [0]` → `1`.

**Ví dụ 2:** `arr = [1,1,2]` → `3` (các OR là `1, 1, 2, 1, 3, 3`, có `3` giá trị khác nhau).

**Ví dụ 3:** `arr = [1,2,4]` → `6` (các OR: `1, 2, 3, 4, 6, 7`).

**Ràng buộc:** `1 <= arr.length <= 5·10^4`; `0 <= arr[i] <= 10^9`.

In [1]:
import itertools
import random
import time
from functools import reduce
from operator import or_
from typing import List

examples = [([0], 1), ([1, 1, 2], 3), ([1, 2, 4], 6)]
for a, want in examples:
    print(f"arr = {a} -> {want}")

arr = [0] -> 1
arr = [1, 1, 2] -> 3
arr = [1, 2, 4] -> 6


## 3. Kiến thức nền

### 3.1 OR theo bit và tính "chỉ tăng"

`a | b` bật mọi bit mà `a` hoặc `b` có. Quan trọng: `a | b ≥ a` và các bit của `a` là **tập con** các bit của `a | b`. Nên khi mở rộng một đoạn con (thêm phần tử), OR của nó không bao giờ mất bit.

In [2]:
print(bin(5), bin(3), bin(5 | 3), "| 5|3 >= 5:", 5 | 3 >= 5, "| 5 là tập con bit của 5|3:", (5 | 3) & 5 == 5)

0b101 0b11 0b111 | 5|3 >= 5: True | 5 là tập con bit của 5|3: True


### 3.2 `functools.reduce` và `operator.or_`

`reduce(or_, [a, b, c])` tính `a | b | c`. Dùng cho oracle: OR của một đoạn con bằng định nghĩa.

### 3.3 Số bit giới hạn số giá trị

Mọi `arr[i] ≤ 10^9 < 2^30` nên có tối đa `30` bit. Với điểm cuối `i` cố định, đi lùi điểm đầu thì OR là dãy **không giảm theo bao hàm bit**; mỗi lần giá trị **thay đổi** thì số bit `1` tăng ít nhất `1`. Dãy giá trị bắt đầu từ `arr[i]` (số bit `1` từ `0` đến `30`) và mỗi lần đổi giá trị thì số bit `1` tăng ít nhất một, nên có tối đa `31` giá trị khác nhau (số bit `1` nhận các giá trị `0, 1, …, 30`).

### 3.4 Duyệt bit

`(x >> b) & 1` lấy bit thứ `b` của `x`. Ta dùng ở Cách 4.

In [3]:
x = 0b101101
print([b for b in range(x.bit_length()) if (x >> b) & 1], "= các bit bật của", bin(x))

[0, 2, 3, 5] = các bit bật của 0b101101


## 4. Ý tưởng nền tảng / trực giác

`arr = [1, 2, 4]`. Với mỗi điểm cuối `i`, liệt kê OR của các đoạn kết thúc tại `i` (điểm đầu lùi dần):

| `i` | các đoạn | OR | tập |
|---|---|---|---|
| 0 | `[1]` | 1 | `{1}` |
| 1 | `[2]`, `[1,2]` | 2, 3 | `{2, 3}` |
| 2 | `[4]`, `[2,4]`, `[1,2,4]` | 4, 6, 7 | `{4, 6, 7}` |

Hợp của các tập: `{1, 2, 3, 4, 6, 7}` có `6` phần tử. Quan sát: tập ở `i` suy ra từ tập ở `i − 1` bằng cách **OR mỗi phần tử với `arr[i]`** rồi thêm chính `arr[i]` (đoạn chỉ một phần tử). Không cần nhớ các đoạn, chỉ nhớ **tập giá trị**.

In [4]:
arr = [1, 2, 4]
for i in range(len(arr)):
    ors = [reduce(or_, arr[j:i + 1]) for j in range(i, -1, -1)]           # điểm đầu lùi dần
    print(f"i={i}: OR của các đoạn kết thúc tại i (điểm đầu lùi dần) = {ors} -> tập {sorted(set(ors))}")

i=0: OR của các đoạn kết thúc tại i (điểm đầu lùi dần) = [1] -> tập [1]
i=1: OR của các đoạn kết thúc tại i (điểm đầu lùi dần) = [2, 3] -> tập [2, 3]
i=2: OR của các đoạn kết thúc tại i (điểm đầu lùi dần) = [4, 6, 7] -> tập [4, 6, 7]


## 5. Cách 1 — Brute Force: mọi đoạn con

**Ý tưởng:** hai vòng lặp `i` (điểm đầu) và `j ≥ i` (điểm cuối), duy trì OR chạy `cur |= arr[j]`, ghi vào `set` toàn cục. Đáp án là kích thước tập.

In [5]:
def or_brute(arr):
    seen = set()
    for i in range(len(arr)):
        cur = 0
        for j in range(i, len(arr)):
            cur |= arr[j]                                # OR của arr[i..j]
            seen.add(cur)
    return len(seen)


for a, want in examples:
    assert or_brute(a) == want
print([or_brute(a) for a, _ in examples])

[1, 3, 6]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(n^2): TLE với n = 5*10^4)
class SolutionBrute:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        seen = set()
        for i in range(len(arr)):
            cur = 0
            for j in range(i, len(arr)):
                cur |= arr[j]
                seen.add(cur)
        return len(seen)

**Độ phức tạp:** `O(n²)` thời gian (với `n = 5·10^4` là `1.25·10^9` phép), `O(số giá trị khác nhau)` bộ nhớ.

**Nút thắt:** ta tính lại OR của nhiều đoạn cho cùng điểm cuối. Nhưng chúng chỉ có `≤ 31` giá trị khác nhau — ta chỉ cần giữ tập giá trị.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `cur_i` là tập OR của mọi đoạn con **kết thúc tại `i`**. Khi thêm `arr[i]`:

`cur_i = { y | arr[i] : y ∈ cur_{i−1} } ∪ { arr[i] }`.

Vì `|cur_i| ≤ 31`, mỗi bước tốn `O(31)`. Đáp án là `| ⋃ cur_i |`. Tổng `O(n · 31)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** lăn tập, **Cách 3** danh sách tăng dần khử trùng liền kề, **Cách 4** "vị trí gần nhất có bit `b`".

### Cách 2 — Lăn tập (hash set)

**Giải thích từng đoạn:**

1. `ans = set()`, `cur = set()` (tập OR các đoạn kết thúc ở vị trí trước).
2. Với mỗi `x`: `cur = {y | x for y in cur} | {x}`; rồi `ans |= cur`.
3. Trả `len(ans)`.

In [7]:
def or_rolling_set(arr):
    ans, cur = set(), set()
    for x in arr:
        cur = {y | x for y in cur} | {x}                 # các đoạn kết thúc tại vị trí hiện tại
        ans |= cur
    return len(ans)


for a, want in examples:
    assert or_rolling_set(a) == want
print([or_rolling_set(a) for a, _ in examples])

[1, 3, 6]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionSet:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ans, cur = set(), set()
        for x in arr:
            cur = {y | x for y in cur} | {x}
            ans |= cur
        return len(ans)

**Độ phức tạp:** `O(n log M)` thời gian (`log M ≈ 30`), `O(n log M)` bộ nhớ trong trường hợp xấu cho `ans` (tới `~31n` giá trị khác nhau), `O(log M)` cho `cur`.

### Cách 3 — Danh sách tăng dần, khử trùng liền kề

**Ý tưởng:** biểu diễn `cur` bằng **danh sách theo thứ tự điểm đầu lùi dần** (tức OR không giảm). Khi thêm `x`: danh sách mới bắt đầu bằng `x`, rồi lần lượt `x | y` cho `y` trong danh sách cũ; giá trị bằng giá trị liền trước thì bỏ (vì dãy không giảm, trùng chỉ xảy ra liền kề). Cách này không cần băm để khử trùng cho `cur`.

In [9]:
def or_monotone_list(arr):
    ans = set()
    cur = []                                             # OR của các đoạn kết thúc tại vị trí trước, điểm đầu lùi dần: không giảm
    for x in arr:
        new = [x]
        for y in cur:
            v = y | x
            if v != new[-1]:                             # dãy không giảm nên chỉ cần so với phần tử liền trước
                new.append(v)
        cur = new
        ans.update(cur)
    return len(ans)


for a, want in examples:
    assert or_monotone_list(a) == want
print([or_monotone_list(a) for a, _ in examples])

[1, 3, 6]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionList:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ans, cur = set(), []
        for x in arr:
            new = [x]
            for y in cur:
                v = y | x
                if v != new[-1]:
                    new.append(v)
            cur = new
            ans.update(cur)
        return len(ans)

**Độ phức tạp:** `O(n log M)` thời gian; `cur` luôn là dãy **tăng thật sự** (nhờ khử trùng) nên độ dài `≤ 31`.

### Cách 4 — Vị trí gần nhất có bit `b`

**Ý tưởng:** OR của `arr[k..i]` được xác định bởi tập bit `b` mà **tồn tại** vị trí `p ∈ [k, i]` có bit `b`, tức `last[b] ≥ k` với `last[b]` là vị trí gần nhất (tính tới `i`) của một phần tử có bit `b`. Giá trị OR chỉ thay đổi khi `k` vượt qua một `last[b]`. Do đó các OR của đoạn kết thúc tại `i` là: với mỗi giá trị `k` thuộc `{last[b]}` ∪ `{i}`, `OR(arr[k..i])` = OR của bit `b` có `last[b] ≥ k`. Sắp xếp các vị trí phân biệt giảm dần và cộng dồn bit.

In [11]:
def or_last_bit(arr):
    ans = set()
    last = [-1] * 31                                     # last[b] = vị trí gần nhất có bit b (hoặc -1)
    for i, x in enumerate(arr):
        for b in range(31):
            if (x >> b) & 1:
                last[b] = i
        # các điểm đầu k có thể làm OR đổi giá trị: i và mọi last[b] (>= 0), giảm dần
        cuts = sorted({i} | {p for p in last if p >= 0}, reverse=True)
        for k in cuts:
            v = 0
            for b in range(31):
                if last[b] >= k:                         # bit b xuất hiện trong arr[k..i]
                    v |= 1 << b
            ans.add(v)
    return len(ans)


for a, want in examples:
    assert or_last_bit(a) == want
print([or_last_bit(a) for a, _ in examples])

[1, 3, 6]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionLastBit:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ans = set()
        last = [-1] * 31
        for i, x in enumerate(arr):
            for b in range(31):
                if (x >> b) & 1:
                    last[b] = i
            for k in {i} | {p for p in last if p >= 0}:
                v = 0
                for b in range(31):
                    if last[b] >= k:
                        v |= 1 << b
                ans.add(v)
        return len(ans)

**Độ phức tạp:** `O(n · 31 · 31)` thời gian (có thể tối ưu bằng cộng dồn), `O(31)` bộ nhớ ngoài `ans`. Ý nghĩa: cho thấy **vì sao** chỉ có `≤ 31` giá trị: chúng ứng với `≤ 31` vị trí `last[b]`.

## 8. TRACE từng bước

**Trace Cách 2** với `arr = [1, 1, 2]`:

| `x` | `cur` trước | `{y\|x}` | `cur` sau | `ans` sau |
|---|---|---|---|---|
| 1 | `{}` | `{}` | `{1}` | `{1}` |
| 1 | `{1}` | `{1}` | `{1}` | `{1}` |
| 2 | `{1}` | `{3}` | `{2, 3}` | `{1, 2, 3}` |

Đáp án `3` ✓. Hàm trace:

In [13]:
def trace_rolling(arr):
    ans, cur = set(), set()
    print(f"arr = {arr}")
    print(f"{'x':>3} {'cur trước':<16} {'{y|x}':<12} {'cur sau':<14} ans sau")
    for x in arr:
        before = sorted(cur)
        ored = sorted({y | x for y in cur})
        cur = {y | x for y in cur} | {x}
        ans |= cur
        print(f"{x:>3} {str(before):<16} {str(ored):<12} {str(sorted(cur)):<14} {sorted(ans)}")
    print("=> số giá trị khác nhau:", len(ans))
    return len(ans)


assert trace_rolling([1, 1, 2]) == 3
print()
assert trace_rolling([1, 2, 4]) == 6

arr = [1, 1, 2]
  x cur trước        {y|x}        cur sau        ans sau
  1 []               []           [1]            [1]
  1 [1]              [1]          [1]            [1]
  2 [1]              [3]          [2, 3]         [1, 2, 3]
=> số giá trị khác nhau: 3

arr = [1, 2, 4]
  x cur trước        {y|x}        cur sau        ans sau
  1 []               []           [1]            [1]
  2 [1]              [3]          [2, 3]         [1, 2, 3]
  4 [2, 3]           [6, 7]       [4, 6, 7]      [1, 2, 3, 4, 6, 7]
=> số giá trị khác nhau: 6


**Trace Cách 4** cho `arr = [1, 2, 4]`: tại `i = 2` (`x = 4`): `last[0] = 0` (bit của `1`), `last[1] = 1`, `last[2] = 2`. Các điểm đầu `k ∈ {2, 1, 0}`: `k = 2` → chỉ bit 2 → `4`; `k = 1` → bit 1, 2 → `6`; `k = 0` → bit 0, 1, 2 → `7`. Khớp tập `{4, 6, 7}` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 2** (lăn tập). Giữ nguyên `class Solution`, `subarrayBitwiseORs(self, arr)`:

In [14]:
class Solution:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ans, cur = set(), set()                           # ans: mọi OR từng thấy; cur: OR các đoạn kết thúc ở vị trí hiện tại
        for x in arr:
            cur = {y | x for y in cur} | {x}              # kéo dài các đoạn cũ thêm x, cộng đoạn [x]
            ans |= cur
        return len(ans)


print([Solution().subarrayBitwiseORs(a) for a, _ in examples])        # [1, 3, 6]
assert all(Solution().subarrayBitwiseORs(a) == want for a, want in examples)

[1, 3, 6]


## 10. Vì sao đúng

**(a) Truy hồi của `cur`.** Đoạn con kết thúc tại `i` hoặc là `[arr[i]]` (độ dài 1), hoặc là `[arr[k..i−1]]` mở rộng thêm `arr[i]`, với OR `= OR(arr[k..i−1]) | arr[i]`. Do đó `cur_i = {arr[i]} ∪ {y | arr[i] : y ∈ cur_{i−1}}` — đúng bằng tập OR của các đoạn kết thúc tại `i`. Hợp qua mọi `i` là tập OR của mọi đoạn con.

**(b) Cỡ của `cur`.** Xếp các đoạn kết thúc tại `i` theo điểm đầu `k = i, i−1, …, 0`; OR `v_k` thoả `v_k ⊆ v_{k−1}` (tập bit). Hai giá trị khác nhau liên tiếp hơn kém nhau ít nhất một bit, mà `v_i = arr[i]` có `≥ 0` bit và mọi giá trị nằm trong `[0, 2^30)` có tối đa `30` bit ⇒ số giá trị khác nhau `≤ 31`.

**(c) Cách 3.** Dãy `new` giữ đúng thứ tự `k` giảm: phần tử đầu `x`, rồi `x | y` theo thứ tự các `y` cũ (không giảm), nên các giá trị bằng nhau là liền kề; bỏ trùng liền kề cho tập `cur` đúng (có cùng tập giá trị với Cách 2).

**(d) Cách 4.** `OR(arr[k..i])` có bit `b` ⇔ tồn tại `p ∈ [k, i]` có bit `b` ⇔ `last[b] ≥ k` (với `last[b]` tính tới `i`). Giá trị chỉ đổi khi `k` đi qua một `last[b]`, nên xét `k ∈ {last[b]} ∪ {i}` đủ để thu mọi giá trị.

Ta kiểm chứng bằng máy: mọi cách khớp **vét cạn mọi đoạn con** (`reduce(or_, …)`) trên mọi mảng nhỏ (mục 12D) và khớp nhau trên mảng lớn; đồng thời `|cur| ≤ 31` được kiểm tra trực tiếp (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi đoạn con | `O(n²)` | `O(D)` | oracle; `D` số giá trị khác nhau |
| 2. Lăn tập | `O(n log M)` | `O(n log M)` | **đáp án chính** |
| 3. Danh sách tăng dần | `O(n log M)` | `O(n log M)` | không cần băm cho `cur` |
| 4. Vị trí gần nhất có bit | `O(n log² M)` | `O(D + log M)` | giải thích giới hạn 31 |

Đo thực tế (`n = 5·10^4`; cách 1 chỉ với `n` nhỏ):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(898)
big = [rng.randint(0, 10 ** 9) for _ in range(50000)]
pow2 = [1 << (i % 30) for i in range(50000)]
for label, data in [("ngẫu nhiên", big), ("luỹ thừa 2 vòng", pow2)]:
    row = [f"{name} {bench(fn, data) * 1e3:8.1f} ms" for name, fn in
           [("set", or_rolling_set), ("list", or_monotone_list), ("lastbit", or_last_bit)]]
    print(f"{label:16s}: " + " | ".join(row))
small = big[:2000]
print(f"cách 1 với n = 2000: {bench(or_brute, small, 1) * 1e3:8.1f} ms (tăng theo n²)")

ngẫu nhiên      : set     77.0 ms | list     41.3 ms | lastbit    745.6 ms


luỹ thừa 2 vòng : set    206.5 ms | list    128.5 ms | lastbit   2272.1 ms
cách 1 với n = 2000:    155.6 ms (tăng theo n²)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle `reduce`, (E) ngẫu nhiên có seed, (F) giới hạn (`n = 5·10^4`, kiểm tra `|cur| ≤ 31`).

### A. Ví dụ của đề

In [16]:
def oracle(arr):
    """Đúng định nghĩa: OR của MỌI đoạn con [i, j] bằng reduce, đếm giá trị khác nhau."""
    n = len(arr)
    return len({reduce(or_, arr[i:j + 1]) for i in range(n) for j in range(i, n)})


IMPLS = {
    "brute": or_brute,
    "set": or_rolling_set,
    "list": or_monotone_list,
    "lastbit": or_last_bit,
    "Solution": lambda a: Solution().subarrayBitwiseORs(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().subarrayBitwiseORs(list(a)),
    "SolutionSet": lambda a: SolutionSet().subarrayBitwiseORs(list(a)),
    "SolutionList": lambda a: SolutionList().subarrayBitwiseORs(list(a)),
    "SolutionLastBit": lambda a: SolutionLastBit().subarrayBitwiseORs(list(a)),
}


def check(arr, want=None, names=None):
    if want is None:
        want = oracle(arr)
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr))
        assert got == want, (name, arr, got, want)


for a, want in examples:
    assert oracle(a) == want
    check(a, want)
print("ví dụ đề: oracle reduce và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle reduce và 9 cài đặt đúng


### B. Edge case

- Một phần tử (`0` và số khác `0`).
- Mọi phần tử bằng nhau.
- Toàn số `0`: đáp án `1`.
- Các luỹ thừa của `2` khác nhau: mọi OR đều khác nhau (`n(n+1)/2`).
- Giá trị lớn nhất `10^9` và `2^30 − 1`... (chú ý `10^9 < 2^30`).
- Mảng có `0` xen kẽ.
- Dãy tăng/giảm.

In [17]:
cases = [
    ([5], 1), ([0, 0, 0], 1), ([7, 7, 7, 7], 1), ([1, 2, 4, 8], 10), ([1, 2, 4, 8, 16], 15),
    ([10 ** 9], 1), ([10 ** 9, 0, 10 ** 9], 2), ([0, 1, 0, 1], 2), ([1, 0, 2, 0, 4], None),
    ([3, 5, 6], None), ([1, 3, 7, 15], 4), ([15, 7, 3, 1], 4), ([2 ** 29, 1, 2 ** 29], None),
]
for a, want in cases:
    check(a, want)
assert Solution().subarrayBitwiseORs([1, 2, 4, 8, 16, 32]) == 21
print(len(cases), "edge case: đúng")

13 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_count_subarrays(arr):                          # đếm số đoạn con, không phải số OR khác nhau
    return len(arr) * (len(arr) + 1) // 2


def bug_and(arr):                                      # dùng & thay vì |
    ans, cur = set(), set()
    for x in arr:
        cur = {y & x for y in cur} | {x}
        ans |= cur
    return len(ans)


def bug_no_self(arr):                                  # quên thêm chính arr[i] (đoạn một phần tử)
    ans, cur = set(), {0}
    for x in arr:
        cur = {y | x for y in cur}                     # {0 | x} có x, nhưng sau lần đầu 0 biến mất
        ans |= cur
    return len(ans)


def bug_zero_in_ans(arr):                              # khởi tạo ans chứa 0
    ans, cur = {0}, set()
    for x in arr:
        cur = {y | x for y in cur} | {x}
        ans |= cur
    return len(ans)


def bug_last_only(arr):                                # chỉ đếm tập của vị trí cuối
    cur = set()
    for x in arr:
        cur = {y | x for y in cur} | {x}
    return len(cur)


def bug_sum_not_or(arr):                               # cộng thay vì OR
    ans, cur = set(), set()
    for x in arr:
        cur = {y + x for y in cur} | {x}
        ans |= cur
    return len(ans)


BUGS = [bug_count_subarrays, bug_and, bug_no_self, bug_zero_in_ans, bug_last_only, bug_sum_not_or]
rng = random.Random(3)
tests = [[rng.randint(0, 15) for _ in range(rng.randint(1, 6))] for _ in range(600)]
for bug in BUGS:
    bad = next((a for a in tests if bug(a) != oracle(a)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên arr = {bad}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_count_subarrays  sai trên arr = [15, 2, 0, 15, 8]: ra 15, đúng 4
bug_and              sai trên arr = [7, 6, 15, 15, 12]: ra 5, đúng 4
bug_no_self          sai trên arr = [4, 11]: ra 2, đúng 3
bug_zero_in_ans      sai trên arr = [4, 11]: ra 4, đúng 3
bug_last_only        sai trên arr = [4, 11]: ra 2, đúng 3
bug_sum_not_or       sai trên arr = [15, 2, 0, 15, 8]: ra 9, đúng 4


### D. Vét cạn mọi mảng nhỏ

(1) Mọi mảng độ dài `1..5` với giá trị `0..7` (`8 + 64 + 512 + 4 096 + 32 768 = 37 448`); (2) mọi mảng độ dài `6` với giá trị trong `{0, 1, 2, 4}` (`4 096`). Oracle: `reduce` trên mọi đoạn con.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "set", "list", "lastbit", "Solution")
for n, vals in [(1, range(8)), (2, range(8)), (3, range(8)), (4, range(8)), (5, range(8)), (6, (0, 1, 2, 4))]:
    for tup in itertools.product(vals, repeat=n):
        a = list(tup)
        want = oracle(a)
        for name in fast:
            assert IMPLS[name](a) == want, (name, a)
        cnt += 1
print(cnt, "mảng nhỏ: 5 cài đặt khớp oracle reduce;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 8 + 64 + 512 + 4096 + 32768 + 4096

41544 mảng nhỏ: 5 cài đặt khớp oracle reduce; 1.5 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên cỡ vừa với nhiều dạng giá trị (bit thưa, bit dày, nhỏ, lớn). Tính chất: (1) đảo mảng **không đổi** đáp án (tập OR của đoạn con không phụ thuộc chiều đọc); (2) thêm một phần tử vào cuối làm đáp án tăng `0..31`; (3) đáp án `≤ min(n(n+1)/2, 31n)`.

In [20]:
rng = random.Random(20240898)
for _ in range(1500):
    n = rng.randint(1, 40)
    mode = rng.choice(["small", "sparse", "dense", "big"])
    if mode == "small":
        a = [rng.randint(0, 7) for _ in range(n)]
    elif mode == "sparse":
        a = [1 << rng.randint(0, 29) if rng.random() < 0.7 else 0 for _ in range(n)]
    elif mode == "dense":
        a = [rng.getrandbits(30) | rng.getrandbits(30) | rng.getrandbits(30) for _ in range(n)]
    else:
        a = [rng.randint(0, 10 ** 9) for _ in range(n)]
    want = oracle(a) if n <= 25 else or_brute(a)
    for name in ("set", "list", "lastbit", "Solution", "SolutionSet", "SolutionList", "SolutionLastBit"):
        assert IMPLS[name](a) == want, (name, a)
    assert Solution().subarrayBitwiseORs(a[::-1]) == want                                  # (1)
    base = want
    ext = Solution().subarrayBitwiseORs(a + [rng.randint(0, 10 ** 9)])
    assert base <= ext <= base + 31                                                        # (2)
    assert want <= min(n * (n + 1) // 2, 31 * n)                                           # (3)
print("1500 mảng ngẫu nhiên: khớp oracle/brute; 3 tính chất (đối xứng, tăng <= 31, cận trên) đều đúng")

1500 mảng ngẫu nhiên: khớp oracle/brute; 3 tính chất (đối xứng, tăng <= 31, cận trên) đều đúng


### F. Giới hạn: `n = 5·10^4` và kiểm tra `|cur| ≤ 31`

Các cách nhanh khớp nhau trên mảng lớn; ghi nhận **kích thước lớn nhất** của tập `cur` (phải `≤ 31`) trên nhiều dạng dữ liệu, kể cả dạng "xấu" làm `cur` lớn.

In [21]:
def max_cur_size(arr):
    cur, best = set(), 0
    for x in arr:
        cur = {y | x for y in cur} | {x}
        best = max(best, len(cur))
    return best


rng = random.Random(500)
n = 50000
datasets = {
    "ngẫu nhiên": [rng.randint(0, 10 ** 9) for _ in range(n)],
    "luỹ thừa 2 vòng": [1 << (i % 30) for i in range(n)],
    "luỹ thừa 2 ngược": [1 << (29 - i % 30) for i in range(n)],
    "toàn 0": [0] * n,
    "toàn 10^9": [10 ** 9] * n,
    "bit thưa": [1 << rng.randint(0, 29) for _ in range(n)],
}
for name, a in datasets.items():
    res = {k: IMPLS[k](a) for k in ("set", "list", "Solution", "SolutionList")}
    assert len(set(res.values())) == 1, (name, res)
    m = max_cur_size(a)
    assert m <= 31, (name, m)
    print(f"{name:18s}: đáp án {res['set']:>7}, max |cur| = {m}")
assert Solution().subarrayBitwiseORs(datasets["toàn 0"]) == 1
assert Solution().subarrayBitwiseORs(datasets["toàn 10^9"]) == 1
print("max |cur| luôn <= 31; mọi cách khớp nhau")

ngẫu nhiên        : đáp án  128893, max |cur| = 10


luỹ thừa 2 vòng   : đáp án     871, max |cur| = 30


luỹ thừa 2 ngược  : đáp án     871, max |cur| = 30
toàn 0            : đáp án       1, max |cur| = 1
toàn 10^9         : đáp án       1, max |cur| = 1


bit thưa          : đáp án  558887, max |cur| = 30
max |cur| luôn <= 31; mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm số đoạn con thay vì số OR khác nhau

In [22]:
print("[1,1,2]: số đoạn con =", bug_count_subarrays([1, 1, 2]), "| đáp án đúng =", Solution().subarrayBitwiseORs([1, 1, 2]))
assert bug_count_subarrays([1, 1, 2]) == 6 != 3

[1,1,2]: số đoạn con = 6 | đáp án đúng = 3


**Vì sao sai:** đề hỏi số giá trị **khác nhau**. **Đúng:** dùng `set` khử trùng.

### Lỗi 2: dùng `&` thay vì `|`

In [23]:
print("AND:", bug_and([1, 2, 4]), "| đúng:", Solution().subarrayBitwiseORs([1, 2, 4]))
assert bug_and([1, 2, 4]) != 6

AND: 4 | đúng: 6


**Vì sao sai:** `&` cho kết quả **giảm** khi mở rộng, không phải OR. **Đúng:** `|`.

### Lỗi 3: quên đoạn chỉ có một phần tử

Khởi tạo `cur = {0}` rồi chỉ `{y | x}`: sau bước đầu `0` biến mất khỏi `cur`, đoạn bắt đầu tại các vị trí sau không được tính.

In [24]:
bad = next(a for a in tests if bug_no_self(a) != oracle(a))
print(bad, "->", bug_no_self(bad), "| đúng:", oracle(bad))

[4, 11] -> 2 | đúng: 3


**Vì sao sai:** đoạn con có thể **bắt đầu** ở mọi vị trí; mỗi bước phải thêm `{x}`. **Đúng:** `cur = {y | x for y in cur} | {x}`.

### Lỗi 4: đưa `0` vào đáp án khi không có đoạn nào OR bằng `0`

In [25]:
print("[1, 2]: có 0 trong ans ->", bug_zero_in_ans([1, 2]), "| đúng:", Solution().subarrayBitwiseORs([1, 2]))
assert bug_zero_in_ans([1, 2]) == 4 and Solution().subarrayBitwiseORs([1, 2]) == 3

[1, 2]: có 0 trong ans -> 4 | đúng: 3


**Vì sao sai:** OR bằng `0` chỉ khi đoạn toàn số `0`. **Đúng:** `ans` bắt đầu rỗng.

### Lỗi 5: chỉ đếm tập của vị trí cuối

In [26]:
print("chỉ vị trí cuối:", bug_last_only([1, 2, 4]), "| đúng:", Solution().subarrayBitwiseORs([1, 2, 4]))
assert bug_last_only([1, 2, 4]) == 3

chỉ vị trí cuối: 3 | đúng: 6


**Vì sao sai:** các OR của đoạn kết thúc ở những vị trí **trước** cũng tính. **Đúng:** `ans |= cur` ở mỗi bước.

### Lỗi 6 (bonus): cộng thay vì OR

In [27]:
bad = next(a for a in tests if bug_sum_not_or(a) != oracle(a))
print(bad, "-> cộng:", bug_sum_not_or(bad), "| đúng:", oracle(bad))

[15, 2, 0, 15, 8] -> cộng: 9 | đúng: 4


**Vì sao sai:** `1 + 1 = 2` nhưng `1 | 1 = 1`. **Đúng:** toán tử `|`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "mọi đoạn con" + phép toán **đơn điệu theo bao hàm** (OR, AND, GCD, LCM theo ước) → với điểm cuối cố định chỉ có `O(log)` giá trị khác nhau ⇒ **lăn tập giá trị** thay vì duyệt mọi điểm đầu.

**Mẹo:**

1. Hỏi: phép toán khi mở rộng đoạn có tính chất đơn điệu không? (OR chỉ bật bit, AND chỉ tắt bit, GCD chỉ giảm.)
2. Giữ tập giá trị của các đoạn **kết thúc tại `i`**, cập nhật theo `i`.
3. Kích thước tập bị chặn bởi số bit (`≤ 31`) hoặc `log` của giá trị.
4. `ans` là hợp mọi tập: dùng `set`.
5. Oracle: `reduce` trên mọi đoạn con.

**Họ bài cùng pattern:** Find a Value of a Mysterious Function Closest to Target (1521), Number of Subarrays With GCD Equal to K (2447), Longest Subarray With Maximum Bitwise AND (2419), Smallest Subarrays With Maximum Bitwise OR (2411), Count Subarrays with Score Less Than K.

## 15. Biến thể

1. **AND thay OR:** số giá trị AND khác nhau (cùng kỹ thuật; tập các AND giảm theo bao hàm).
2. **Đếm số đoạn con có OR bằng `K`:** dùng `dict` giá trị → số điểm đầu.
3. **Số đoạn con có OR `≥ K`.**
4. **GCD** của đoạn con (số giá trị khác nhau).
5. **Số đoạn con ngắn nhất có OR lớn nhất bắt đầu tại mỗi `i`** (2411).

Cài biến thể 1 và 2 và đối chiếu vét cạn:

In [28]:
def distinct_ands(arr):
    ans, cur = set(), set()
    for x in arr:
        cur = {y & x for y in cur} | {x}
        ans |= cur
    return len(ans)


def distinct_ands_brute(arr):
    n = len(arr)
    return len({reduce(lambda p, q: p & q, arr[i:j + 1]) for i in range(n) for j in range(i, n)})


def count_subarrays_or_equal(arr, K):
    """Số đoạn con có OR đúng bằng K: lăn dict {giá trị OR: số điểm đầu cho giá trị đó}."""
    total, cur = 0, {}
    for x in arr:
        nxt = {x: 1}
        for y, c in cur.items():
            v = y | x
            nxt[v] = nxt.get(v, 0) + c
        cur = nxt
        total += cur.get(K, 0)
    return total


def count_subarrays_or_equal_brute(arr, K):
    n = len(arr)
    return sum(1 for i in range(n) for j in range(i, n) if reduce(or_, arr[i:j + 1]) == K)


rng = random.Random(15)
for _ in range(600):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 10))]
    assert distinct_ands(a) == distinct_ands_brute(a)
    K = rng.randint(0, 15)
    assert count_subarrays_or_equal(a, K) == count_subarrays_or_equal_brute(a, K)
print("distinct ANDs và đếm đoạn con có OR = K khớp vét cạn trên 600 mảng")
assert count_subarrays_or_equal([1, 1, 2], 3) == 2 and count_subarrays_or_equal([1, 1, 2], 1) == 3

distinct ANDs và đếm đoạn con có OR = K khớp vét cạn trên 600 mảng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn con | OR chạy với hai vòng lặp | `O(n²)` | `O(D)` | oracle, `n` nhỏ |
| 2. Lăn tập | `cur = {y\|x} ∪ {x}` | `O(n log M)` | `O(n log M)` | **đáp án chính** |
| 3. Danh sách tăng dần | khử trùng liền kề | `O(n log M)` | `O(n log M)` | không băm cho `cur` |
| 4. Vị trí gần nhất có bit | cắt theo `last[b]` | `O(n log² M)` | `O(D)` | giải thích `≤ 31` |

Cả bốn khớp oracle vét cạn trên 37 448 + 4 096 mảng nhỏ và khớp nhau trên `n = 5·10^4` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().subarrayBitwiseORs([3, 5]), Solution().subarrayBitwiseORs([1, 3, 7]), Solution().subarrayBitwiseORs([4, 4, 0, 4]), distinct_ands([6, 3, 5]))

3 3 2 6


### Bài 2 (tính tay)

`arr = [2, 1, 4]`. (a) Với mỗi điểm cuối `i`, liệt kê tập OR các đoạn kết thúc tại `i`. (b) Hợp của chúng và đáp án? (c) Có bao nhiêu đoạn con có OR bằng `7`? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện lăn tập:

In [30]:
def rolling_todo(arr):
    ans, cur = set(), set()
    for x in arr:
        pass            # TODO 1: cur = tập OR của các đoạn kết thúc tại x (kéo dài cũ rồi cộng {x})
        pass            # TODO 2: ans |= cur
    return len(ans)


# Khi làm xong, bỏ comment để kiểm tra:
# assert [rolling_todo(a) for a, _ in examples] == [1, 3, 6]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **danh sách tăng dần khử trùng liền kề**:

In [31]:
class MySolution:
    def subarrayBitwiseORs(self, arr):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().subarrayBitwiseORs(a) == w for a, w in examples)

### Bài 5 (mini challenge): đoạn con ngắn nhất cho OR lớn nhất (2411)

Với mỗi `i`, tìm độ dài **ngắn nhất** của đoạn con bắt đầu tại `i` có OR bằng OR lớn nhất có thể của mọi đoạn con bắt đầu tại `i`. Viết `smallest_subarrays(arr)` trong `O(n log M)` (gợi ý: duyệt từ phải sang trái, giữ `last[b]` = vị trí gần nhất bên phải có bit `b`). Kiểm tra bằng vét cạn.

In [32]:
def smallest_subarrays(arr):
    pass            # TODO


# Khi làm xong:
# assert smallest_subarrays([1, 0, 2, 1, 3]) == [3, 3, 2, 2, 1]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: với mỗi điểm cuối, kéo dài các OR cũ.
- Mức 2: `[3,5]`: `{3}`, rồi `{5, 3|5 = 7}` → `{3,5,7}` = `3`.
- Mức 3: kết quả `3 3 2 6`: `[1,3,7]` cho `{1, 3, 7}`; `[4,4,0,4]` cho `{4, 0}`; AND của `[6,3,5]`: `{6}`, `{3, 2}`, `{5, 1, 0}` → hợp `{6,3,2,5,1,0}` có `6` giá trị.

**Bài 2**

- Mức 1: `i=0`: `{2}`.
- Mức 2: `i=1`: `{1, 3}`; `i=2`: `{4, 5, 7}`.
- Mức 3: hợp `{1,2,3,4,5,7}` → `6`. OR bằng `7` chỉ có đoạn `[2,1,4]` (còn `[1,4]` cho `5`, `[2,1]` cho `3`) nên có `1` đoạn.

**Bài 3**

- Mức 1: `cur = {y | x for y in cur} | {x}`.
- Mức 2: `ans |= cur`.
- Mức 3: xem Cách 2.

**Bài 4**

- Mức 1: `new = [x]`.
- Mức 2: duyệt `y` trong `cur`, `v = y | x`, thêm nếu `v != new[-1]`.
- Mức 3: `cur = new`, `ans.update(cur)`.

**Bài 5**

- Mức 1: duyệt `i` từ phải sang trái, cập nhật `last[b] = i` với mỗi bit của `arr[i]`.
- Mức 2: đoạn bắt đầu tại `i` cần tới `max(last[b])` (bỏ `-1`, tối thiểu `i`).
- Mức 3: độ dài `max(1, max_last - i + 1)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().subarrayBitwiseORs([3, 5]), Solution().subarrayBitwiseORs([1, 3, 7]), Solution().subarrayBitwiseORs([4, 4, 0, 4]), distinct_ands([6, 3, 5]))
assert vals == (3, 3, 2, 6) == (oracle([3, 5]), oracle([1, 3, 7]), oracle([4, 4, 0, 4]), distinct_ands_brute([6, 3, 5]))
print("Bài 1 OK:", vals)

Bài 1 OK: (3, 3, 2, 6)


### Bài 2

`i=0`: `{2}`; `i=1`: `{1, 3}`; `i=2`: `{4, 5, 7}`. Hợp `{1,2,3,4,5,7}` ⇒ đáp án `6`. Chỉ `[2,1,4]` có OR `7`.

In [34]:
a = [2, 1, 4]
sets = []
cur = set()
for x in a:
    cur = {y | x for y in cur} | {x}
    sets.append(sorted(cur))
assert sets == [[2], [1, 3], [4, 5, 7]]
assert Solution().subarrayBitwiseORs(a) == 6 and count_subarrays_or_equal(a, 7) == 1
print("Bài 2 OK:", sets)

Bài 2 OK: [[2], [1, 3], [4, 5, 7]]


### Bài 3

In [35]:
def rolling_answer(arr):
    ans, cur = set(), set()
    for x in arr:
        cur = {y | x for y in cur} | {x}
        ans |= cur
    return len(ans)


assert [rolling_answer(a) for a, _ in examples] == [1, 3, 6]
for a in tests[:300]:
    assert rolling_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def subarrayBitwiseORs(self, arr):
        ans, cur = set(), []
        for x in arr:
            new = [x]
            for y in cur:
                v = y | x
                if v != new[-1]:
                    new.append(v)
            cur = new
            ans.update(cur)
        return len(ans)


assert all(MySolutionAnswer().subarrayBitwiseORs(a) == w for a, w in examples)
for a in tests[:300]:
    assert MySolutionAnswer().subarrayBitwiseORs(a) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def smallest_subarrays_answer(arr):
    n = len(arr)
    last = [-1] * 31
    res = [0] * n
    for i in range(n - 1, -1, -1):
        for b in range(31):
            if (arr[i] >> b) & 1:
                last[b] = i
        far = max(max(last), i)
        res[i] = far - i + 1
    return res


def smallest_subarrays_brute(arr):
    n = len(arr)
    res = []
    for i in range(n):
        best_or = reduce(or_, arr[i:])
        for j in range(i, n):
            if reduce(or_, arr[i:j + 1]) == best_or:
                res.append(j - i + 1)
                break
    return res


assert smallest_subarrays_answer([1, 0, 2, 1, 3]) == [3, 3, 2, 2, 1] == smallest_subarrays_brute([1, 0, 2, 1, 3])
rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 10))]
    assert smallest_subarrays_answer(a) == smallest_subarrays_brute(a)
print("Bài 5 OK: khớp vét cạn")

Bài 5 OK: khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (phép toán đơn điệu trên mọi đoạn con → lăn tập giá trị của các đoạn kết thúc tại `i`):**

```python
class Solution:
    def subarrayBitwiseORs(self, arr: List[int]) -> int:
        ans, cur = set(), set()
        for x in arr:
            cur = {y | x for y in cur} | {x}   # kéo dài đoạn cũ + đoạn [x]
            ans |= cur
        return len(ans)
```

**Checklist trước khi submit:**

- [ ] Dùng `|` (không `&`, không `+`)?
- [ ] `cur` luôn gồm cả `{x}` (đoạn một phần tử)?
- [ ] `ans |= cur` ở **mỗi** bước, `ans` bắt đầu **rỗng**?
- [ ] Đếm số giá trị **khác nhau** (`set`), không phải số đoạn con?
- [ ] `|cur| ≤ 31` nên `O(n log M)`?
- [ ] Đã thử `[0] → 1`, `[1,1,2] → 3`, `[1,2,4] → 6`, toàn số `0`, luỹ thừa của 2?
- [ ] Đúng tên `class Solution`, `subarrayBitwiseORs(self, arr)`, không in debug.